# **Bronze — VRA (Voo Regular Ativo)**
Lê os 12 CSVs mensais do volume voebem.bronze.arquivos/vra/ e materializa voebem.bronze.vra.

Regras da camada Bronze:

- **nada de tipagem** — tudo string, exatamente como veio do arquivo;

- **nada de filtro** — nenhuma linha é descartada;

- **colunas de auditoria** — de qual arquivo veio e quando foi ingerido;

- **idempotente** — rodar duas vezes não duplica.

In [0]:
# Importa as funções do PySpark para manipulação de dados
from pyspark.sql import functions as F

# Define o caminho onde estão os arquivos CSV de VRA no volume Unity Catalog
CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"

# Define o nome completo da tabela de destino (catálogo.schema.tabela)
TABELA = "voebem.bronze.vra"

# Leitura

Quatro opções carregam quatro problemas do arquivo:

| opção | resolve |
| --- | --- |
| `sep=";"` | separador brasileiro, não vírgula |
| `skipRows=1` | a 1ª linha é `Atualizado em: <data>`, não o cabeçalho — e o BOM `EF BB BF` mora nela, some junto |
| `header=true` | a 2ª linha (a primeira que sobra) é o cabeçalho de verdade |
| `inferSchema` **desligado** (default) | bronze não tipa: tudo chega como `string` |

In [0]:
# Lê os arquivos CSV com as configurações específicas do formato VRA da ANAC
bruto = (
    spark.read.format("csv")
    .option("sep", ";")              # Separador brasileiro (ponto-e-vírgula)
    .option("header", True)          # Primeira linha válida contém os nomes das colunas
    .option("skipRows", 1)           # Pula a primeira linha ("Atualizado em: <data>" + BOM UTF-8)
    .option("quote", '"')            # Campos com texto podem estar entre aspas duplas
    .option("escape", '"')           # Escape de aspas duplas dentro de campos
    .option("encoding", "UTF-8")     # Codificação do arquivo
    .option("mode", "PERMISSIVE")    # Modo permissivo: aceita linhas malformadas (coloca null)
    .load(CAMINHO)                   # Carrega todos os CSVs do caminho (padrão glob)
)

# Exibe as colunas lidas do arquivo para validação
print("colunas lidas do arquivo:")
for c in bruto.columns:
    print(f"    {c!r}")
print()

%md
# Nomes de coluna: o Delta não aceita espaço

`ICAO Empresa Aérea` é um nome de coluna válido em CSV e **inválido** em Delta — espaço está na lista de caracteres proibidos (` ,;{}()\n\t= `).

Então normalizamos o **nome**. Repare que isso não fere a regra da bronze: o que a bronze preserva é o **valor** e a **granularidade**, não a grafia do cabeçalho. Nenhuma coluna é somada, removida, filtrada ou convertida.

O mapa fica explícito no código — nada de `regexp_replace` mágico, para que a correspondência com o arquivo original seja auditável.

In [0]:
# Mapa explícito: nome original no CSV -> nome normalizado (snake_case)
# Delta Lake não aceita espaços em nomes de colunas, por isso a normalização
RENOMEAR = {
    "ICAO Empresa Aérea":          "icao_empresa",
    "Número Voo":                  "numero_voo",
    "Código Autorização (DI)":      "codigo_di",
    "Código Tipo Linha":            "codigo_tipo_linha",
    "ICAO Aeródromo Origem":        "icao_aerodromo_origem",
    "ICAO Aeródromo Destino":       "icao_aerodromo_destino",
    "Partida Prevista":             "partida_prevista",
    "Partida Real":                 "partida_real",
    "Chegada Prevista":             "chegada_prevista",
    "Chegada Real":                 "chegada_real",
    "Situação Voo":                 "situacao_voo",
    "Código Justificativa":         "codigo_justificativa",
}

# Valida se todas as colunas esperadas estão presentes no arquivo
# (chaves do dicionário que não apareceram no arquivo - deve ficar vazia)
faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada nao encontrada no CSV: {faltando}"

# Renomeia as colunas e garante que todos os valores sejam do tipo string (regra da bronze)
# Bronze não faz tipagem: tudo permanece como veio do arquivo original
renomeado = bruto.select(
    [F.col(origem).cast("string").alias(novo) for origem, novo in RENOMEAR.items()]
)

%md
# Auditoria

Duas colunas que o arquivo não tem e a tabela precisa ter:
`_arquivo_origem` (de qual CSV a linha veio — `_metadata` é uma coluna oculta que o Spark expõe em qualquer leitura de arquivo) e `_ingerido_em`.

In [0]:
# Adiciona colunas de auditoria necessárias na camada bronze
bronze = renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")  # Nome do arquivo CSV de origem (ex: VRA_20261.csv)
).withColumn(
    "ingerido_em", F.current_timestamp()              # Timestamp de quando o dado foi ingerido
)


%md
# Escrita idempotente

Estratégia: **full refresh determinístico** — `mode("overwrite")` sobre o conjunto inteiro de arquivos.

Por que essa e não um `append` com deduplicação:

1. A fonte é **imutável e completa**: o volume tem os 12 arquivos do mês fechado, e a ANAC republica o mês inteiro quando corrige algo. A entrada define o estado final — logo o destino pode ser derivado inteiro dela.
2. `append` exigiria uma chave de negócio para deduplicar. O VRA **não tem chave natural única** (o mesmo voo pode repetir legitimamente na mesma data — veja o código DI "Etapa de Voo Duplicada"). Deduplicar no bronze seria decidir regra de negócio na camada errada.
3. `overwrite` no Delta é **atômico**: ou a versão nova aparece inteira, ou a antiga continua valendo. Ninguém lê tabela pela metade.
4. O histórico não se perde: cada `overwrite` gera uma versão nova no log do Delta, e a anterior continua acessível por time travel (marco-04).

O que muda entre duas execuções: só `_ingerido_em`. **O conjunto de linhas é idêntico** — é isso que a validação prova.

In [0]:
# Grava a tabela bronze no formato Delta Lake com estratégia de full refresh
# Mode overwrite: substitui toda a tabela (idempotente - pode rodar múltiplas vezes)
(
    bronze.write.format("delta")
    .mode("overwrite")                      # Substitui completamente a tabela existente
    .option("overwriteSchema", "true")      # Permite alteração no schema se necessário
    .saveAsTable(TABELA)                    # Grava como tabela gerenciada do Unity Catalog
)

# Exibe o total de linhas carregadas na tabela para validação
print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

In [0]:
# Adiciona documentação/comentário na tabela do Unity Catalog
# Contextualização de Catálogo - facilita entendimento para outros usuários
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
    Dado bruto: todas as colunas string, nenhuma linha descartada.
    Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/.'           
""")

In [0]:
# Query de validação: exibe quantas linhas vieram de cada arquivo CSV
# Permite verificar se todos os 12 meses foram carregados corretamente
display(
    spark.sql(f"""
            SELECT _arquivo_origem, COUNT(*) AS linhas, MAX
            (ingerido_em) AS ingerido_em
            FROM {TABELA}
            GROUP BY _arquivo_origem
            ORDER BY _arquivo_origem  
        """)
)